# Run the existing YOLO + best84 pipeline from GitHub

Run the cells in order on a Colab **T4 GPU** runtime. This notebook does not train models or submit to Kaggle automatically. It retrieves source code from GitHub and uses the existing images and checkpoints in your Drive.

It generates both the original submission baseline CSV and an R1 comparison CSV. Both pipelines have a reported public leaderboard score of 0.39; an R1 leaderboard improvement has not been confirmed. Rerunning the notebook reuses the saved code commit and completed caches. Use a new OUTPUT directory to compare a different commit or configuration.

The notebook syntax and setup have been checked. A complete GPU run through this new GitHub entry point has not yet been verified. If the repository is private, Colab requires separate GitHub authentication. Do not save tokens in the notebook.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, importlib.util, json, os, signal, subprocess, sys, time

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/filament-segmentation-2026')
TEST_IMAGES = BASE/'MAGFiLO_1.0_Kaggle_2026/test/test_images'
YOLO_CHECKPOINT = Path('/content/drive/MyDrive/SolarFilament_LimbOnly_YOLO26_mixed_FullRes/best.pt')
NATIVE_CHECKPOINT = Path('/content/drive/MyDrive/SolarFilament_NativeUNet_A100/20261028T030808Z_bedf3e21/best.pt')
OUTPUT = BASE/'submissions/github_best84_t4_v1'
MAX_IMAGES = None  # Set to 2 for a quick check, then to None to resume processing all 180 images.
REPOSITORY = 'https://github.com/ikjoo5/solar_filament.git'
REF = 'main'  # Used only on the first run. Resume uses the commit saved in OUTPUT.
CODE = Path('/content/solar_filament_github')


In [ ]:
def run_git(*args):
    return subprocess.check_output(['git', *args], text=True,
        env=dict(os.environ, GIT_TERMINAL_PROMPT='0')).strip()

OUTPUT.mkdir(parents=True, exist_ok=True)
pin_path = OUTPUT/'source_commit.json'
pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
if pin and pin['repository'] != REPOSITORY:
    raise ValueError('The saved repository URL differs. Use a separate OUTPUT directory.')
if not CODE.exists():
    run_git('clone', '--no-checkout', REPOSITORY, str(CODE))
if run_git('-C', str(CODE), 'remote', 'get-url', 'origin') != REPOSITORY:
    raise ValueError('The CODE directory contains a different repository.')
# After --no-checkout, the index is empty; skip the status check before the first checkout.
if (CODE/'.git/index').exists() and run_git('-C', str(CODE), 'status', '--porcelain'):
    raise ValueError('Local code changes are preserved. Choose a different CODE directory.')
revision = pin['commit'] if pin else REF
run_git('-C', str(CODE), 'fetch', '--depth', '1', 'origin', revision)
commit = run_git('-C', str(CODE), 'rev-parse', 'FETCH_HEAD')
if pin and commit != pin['commit']:
    raise ValueError('The code commit does not match the saved resume commit.')
run_git('-C', str(CODE), 'checkout', '--detach', commit)
if pin is None:
    pending = pin_path.with_suffix('.pending')
    pending.write_text(json.dumps(dict(repository=REPOSITORY, commit=commit), indent=2))
    pending.replace(pin_path)
print('Pinned code commit:', commit)

spec = importlib.util.spec_from_file_location('repo_bootstrap', CODE/'tools/r1_submission_colab_bootstrap.py')
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
for protected in (TEST_IMAGES, YOLO_CHECKPOINT.parent, NATIVE_CHECKPOINT.parent):
    bootstrap.require_separate(OUTPUT, protected)
for path in (YOLO_CHECKPOINT, NATIVE_CHECKPOINT):
    if not path.is_file(): raise FileNotFoundError(str(path))
if not TEST_IMAGES.is_dir(): raise FileNotFoundError(str(TEST_IMAGES))
bootstrap.ensure_dependencies(OUTPUT)  # Preserve Colab's installed torch/torchvision.

names = sorted(p.name for p in TEST_IMAGES.rglob('*')
    if p.is_file() and p.suffix.lower() in ('.jpeg', '.jpg', '.png'))
expected = 'f9b2272771d3f6666f6ef812f79968a27f3d448ac2f277eda46c854ad38d598f'
encoded = json.dumps(names, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode()
if len(names) != 180 or len(set(names)) != 180 or hashlib.sha256(encoded).hexdigest() != expected:
    raise ValueError('The filenames do not match the official test180 list.')

def immutable_json(path, value):
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError('The existing configuration is preserved. Use a separate OUTPUT directory: '+str(path))
    else:
        pending = path.with_suffix('.pending')
        pending.write_text(json.dumps(value, ensure_ascii=False, indent=2))
        pending.replace(path)

names_path = OUTPUT/'official_test_names.json'
immutable_json(names_path, names)
config = dict(images=str(TEST_IMAGES), expected_names_path=str(names_path),
    yolo_checkpoint=str(YOLO_CHECKPOINT), native_checkpoint=str(NATIVE_CHECKPOINT),
    output_root=str(OUTPUT), device='cuda', native_batch=1, precision='fp16', expected_images=180)
config_path = OUTPUT/'github_config.json'
immutable_json(config_path, config)
print('Configuration saved:', config_path)


In [ ]:
if MAX_IMAGES is not None and (type(MAX_IMAGES) is not int or not 1 <= MAX_IMAGES <= 180):
    raise ValueError('MAX_IMAGES must be None or an integer from 1 to 180.')
command = [sys.executable, '-u', str(CODE/'tools/r1_test_submission.py'), '--config', str(config_path)]
if MAX_IMAGES is not None: command += ['--max-images', str(MAX_IMAGES)]
env = dict(os.environ, PYTHONUNBUFFERED='1', TORCH_CUDNN_V8_API_DISABLED='1',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1')
log_path = OUTPUT/'github_inference.log'
def tail(count=5):
    with log_path.open('rb') as stream:
        stream.seek(max(0, log_path.stat().st_size-24000))
        return '\n'.join(stream.read().decode('utf-8', 'replace').splitlines()[-count:])
print('Full execution log:', log_path)
with log_path.open('ab', buffering=0) as log:
    process = subprocess.Popen(command, cwd=CODE, env=env, stdout=log,
        stderr=subprocess.STDOUT, start_new_session=True)
    try:
        while True:
            try:
                code = process.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                print(tail(), flush=True)
        if code:
            print(tail(50))
            raise subprocess.CalledProcessError(code, command)
    finally:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
result = json.loads((OUTPUT/'latest_results.json').read_text())
print('Status:', result['status'], 'Completed images:', result['completed_images'])
if result['status'] == 'complete':
    manifest = json.loads((OUTPUT/'run_manifest.json').read_text())
    for name, expected_sha in manifest['output_sha256'].items():
        if hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest() != expected_sha:
            raise ValueError('A completed output file has changed: '+name)
    if hashlib.sha256(Path(result['archive']).read_bytes()).hexdigest() != result['archive_sha256']:
        raise ValueError('The completed results ZIP has changed.')
    for key in ('baseline_csv', 'r1_csv', 'archive'):
        path = Path(result[key])
        print('Output:', path)
        try: files.download(str(path))
        except Exception: print('You can download the file directly from Drive:', path)
else:
    print('This was a partial run. Rerun this notebook with MAX_IMAGES=None.')
